In [ ]:
from pathlib import Path
import os
import sys
import json

import torch
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

from groundingdino.util.inference import load_model

ROOT = Path.cwd().parent
sys.path.append(str(ROOT))
from src.common.nuscenes_utils import (
    get_scene_contents,
    get_sample_contents,
    get_sample_data_bboxes,
    CATEGORY_MAPPING_TO_UNIAD
)
from src.common.geometry.detection import (
    convert_global_bbox_to_ego,
    filter_boxes_in_camera_fov,
)
from src.common.image_processing.detection import convert_3d_box_to_2d_box
from src.common.visualize.detection import (
    plot_3d_boxes_on_image,
    plot_2d_boxes_on_image,
)
from src.common.visualize.colors import TABLEAU10_NAMES

from src.grounding_dino.inference import predict_multi_labels

# Resolve paths relative to this notebook directory
CONFIG_PATH = ROOT / "GroundingDINO" / "groundingdino/config/GroundingDINO_SwinB_cfg.py"
WEIGHTS_PATH = ROOT / "GroundingDINO" / "weights/groundingdino_swinb_cogcoor.pth"
device = "cuda" if torch.cuda.is_available() else "cpu"
# Create the model and load the weights
model = load_model(str(CONFIG_PATH), str(WEIGHTS_PATH), device=device)
# Load nuScenes dataset
NUSCENES_ROOT = Path.cwd().parent / "data/nuscenes"
NUSCENES_VERSION = "v1.0-trainval"
with open(NUSCENES_ROOT / NUSCENES_VERSION / "scene.json") as f:
    scenes = json.load(f)
with open(NUSCENES_ROOT / NUSCENES_VERSION / "sample.json") as f:
    samples_all = json.load(f)
with open(NUSCENES_ROOT / NUSCENES_VERSION / "sample_data.json") as f:
    sample_data_all = json.load(f)
with open(NUSCENES_ROOT / NUSCENES_VERSION / "ego_pose.json") as f:
    ego_poses_all = json.load(f)
with open(NUSCENES_ROOT / NUSCENES_VERSION / "calibrated_sensor.json") as f:
    calibrated_sensors_all = json.load(f)
with open(NUSCENES_ROOT / NUSCENES_VERSION / "sensor.json") as f:
    sensors = json.load(f)
with open(NUSCENES_ROOT / NUSCENES_VERSION / "sample_annotation.json") as f:
    sample_annotations_all = json.load(f)
with open(NUSCENES_ROOT / NUSCENES_VERSION / "instance.json") as f:
    instances_all = json.load(f)
with open(NUSCENES_ROOT / NUSCENES_VERSION / "category.json") as f:
    categories = json.load(f)

# Create hash maps for token lookup
sensors = {sensor["token"]: sensor for sensor in sensors}
sensor_lookup = {sensor["channel"]: sensor["token"] for sensor in sensors.values()}

categories = {category["token"]: category for category in categories}
category_conversion = {k: v["category_name"] for k, v in CATEGORY_MAPPING_TO_UNIAD.items()}
category_names = list(dict.fromkeys(
    mapping["category_name"]
    for mapping in sorted(
        CATEGORY_MAPPING_TO_UNIAD.values(),
        key=lambda mapping: mapping["id"],
    )
))

print(f"original_category_names: {[category['name'] for category in categories.values()]}")
print(f"sensor_channels: {[sensor['channel'] for sensor in sensors.values()]}")
print(f"scene_names: {[scene['name'] for scene in scenes]}")

In [ ]:
# Select the scenes and camera channel
SCENE_NAME = "scene-0061"
CAMERA_CHANNEL = "CAM_FRONT"

# Get the scene contents for the selected scene
scene = next(scene for scene in scenes if scene["name"] == SCENE_NAME)
scene_contents = get_scene_contents(scene["token"], samples_all, 
                                    sample_data_all, ego_poses_all, calibrated_sensors_all,
                                    sample_annotations_all=sample_annotations_all,
                                    instances_all=instances_all)
samples = scene_contents["samples"]
sample_data = scene_contents["sample_data"]
ego_poses = scene_contents["ego_poses"]
calibrated_sensors = scene_contents["calibrated_sensors"]
sample_annotations = scene_contents["sample_annotations"]
instances = scene_contents["instances"]
# Create track_ids from instance tokens
track_ids = {inst_token: i for i, inst_token in enumerate(instances.keys())}

# Show the first three sample data entries for the selected scene
for i in range(len(samples)):
    if i >= 3:
        break
    # Get the contents of the current sample and the specified camera channel
    sample_contents_cam = get_sample_contents(i, samples, sample_data, ego_poses, calibrated_sensors,
                                              sensor_token=sensor_lookup[CAMERA_CHANNEL])
    sample_data_cam = list(sample_contents_cam["sample_data"].values())[0]
    image_width, image_height = sample_data_cam["width"], sample_data_cam["height"]
    calibrated_sensor_cam = list(sample_contents_cam["calibrated_sensors"].values())[0]
    camera_translation = calibrated_sensor_cam["translation"]
    camera_rotation = calibrated_sensor_cam["rotation"]
    camera_intrinsic = calibrated_sensor_cam["camera_intrinsic"]
    ego_pose_cam = list(sample_contents_cam["ego_poses"].values())[0]
    # Get the 3D bounding boxes for the sample data
    boxes_3d = get_sample_data_bboxes(sample_data_cam, sample_annotations, instances, categories,
                                      category_conversion=category_conversion,
                                      track_ids=track_ids)
    # Convert the boxes to ego coordinates
    boxes_3d_ego = [convert_global_bbox_to_ego(box, ego_pose_cam["translation"], ego_pose_cam["rotation"]) for box in boxes_3d]
    # Filter the boxes to keep only those that are in the camera's field of view
    valid_boxes_3d_ego = filter_boxes_in_camera_fov(boxes_3d_ego, camera_translation, camera_rotation,
                                                    camera_intrinsic, image_width, image_height)
    print(f"Number of boxes in the camera's field of view: {len(valid_boxes_3d_ego)}")
    # Visulize the 3d boxes in the camera's field of view
    image_path = NUSCENES_ROOT / sample_data_cam["filename"]
    image = Image.open(image_path)
    len_axes = len(category_names) + 1  # +1 for all categories
    num_cols = min(len_axes, 3)
    num_rows = (len_axes + num_cols - 1) // num_cols
    fig, axes = plt.subplots(nrows=num_rows, ncols=num_cols, figsize=(18, 4 * num_rows))
    plot_3d_boxes_on_image(image, valid_boxes_3d_ego, camera_translation, 
                           camera_rotation, camera_intrinsic,
                           ax=axes[0, 0] if num_rows > 1 else axes[0] if num_cols > 1 else axes, title="All Categories")
    # Visualize the 3d boxes for each selected category
    for j, category in enumerate(category_names):
        category_boxes = [box for box in valid_boxes_3d_ego if box.label == category]
        row = (j + 1) // num_cols
        col = (j + 1) % num_cols
        ax = axes[row, col] if num_rows > 1 else axes[col] if num_cols > 1 else axes
        plot_3d_boxes_on_image(image, category_boxes, camera_translation, camera_rotation,
                               camera_intrinsic,
                               ax=ax, title=category)
    fig.suptitle("3D Bounding Boxes, " + CAMERA_CHANNEL + ", sample_token:" + sample_data_cam["sample_token"])
    plt.show()

    # Convert 3d boxes to 2d boxes in the image plane
    boxes_2d = [convert_3d_box_to_2d_box(box, camera_translation, camera_rotation, camera_intrinsic, image_width, image_height)
                for box in valid_boxes_3d_ego]
    boxes_2d_filtered = [box for box in boxes_2d if box is not None]
    print(f"Number of 2D boxes in the image plane: {len(boxes_2d_filtered)}")
    # Plot the 2D boxes on the image
    fig, axes = plt.subplots(nrows=num_rows, ncols=num_cols, figsize=(18, 4 * num_rows))
    plot_2d_boxes_on_image(image, boxes_2d_filtered, ax=axes[0, 0] if num_rows > 1 else axes[0] if num_cols > 1 else axes, 
                           title="All Categories")
    
    for j, category in enumerate(category_names):
        category_boxes = [box for box in boxes_2d_filtered if box.label == category]
        row = (j + 1) // num_cols
        col = (j + 1) % num_cols
        ax = axes[row, col] if num_rows > 1 else axes[col] if num_cols > 1 else axes
        plot_2d_boxes_on_image(image, category_boxes, ax=ax, title=category)
    fig.suptitle("2D Bounding Boxes, " + CAMERA_CHANNEL + ", sample_token:" + sample_data_cam["sample_token"])
    plt.show()

In [ ]:
# Inference with GroundingDINO model
BOX_THRESHOLDS = [0.25, 0.35, 0.45]
# Iterate the first three sample data entries for the selected scene
for i in range(len(samples)):
    if i >= 3:
        break
    # Get the contents of the current sample
    sample_contents_cam = get_sample_contents(i, samples, sample_data, ego_poses, calibrated_sensors,
                                              sensor_token=sensor_lookup[CAMERA_CHANNEL])
    sample_data_cam = list(sample_contents_cam["sample_data"].values())[0]
    image_width, image_height = sample_data_cam["width"], sample_data_cam["height"]
    calibrated_sensor_cam = list(sample_contents_cam["calibrated_sensors"].values())[0]
    camera_translation = calibrated_sensor_cam["translation"]
    camera_rotation = calibrated_sensor_cam["rotation"]
    camera_intrinsic = calibrated_sensor_cam["camera_intrinsic"]
    ego_pose_cam = list(sample_contents_cam["ego_poses"].values())[0]
    # Get the ground truth bounding boxes for the sample data
    boxes_3d = get_sample_data_bboxes(sample_data_cam, sample_annotations, instances, categories,
                                      category_conversion=category_conversion,
                                      track_ids=track_ids)
    boxes_3d_ego = [convert_global_bbox_to_ego(box, ego_pose_cam["translation"], ego_pose_cam["rotation"]) for box in boxes_3d]
    valid_boxes_3d_ego = filter_boxes_in_camera_fov(boxes_3d_ego, camera_translation, camera_rotation,
                                                    camera_intrinsic, image_width, image_height)
    boxes_2d = [convert_3d_box_to_2d_box(box, camera_translation, camera_rotation, camera_intrinsic, image_width, image_height)
                for box in valid_boxes_3d_ego]
    boxes_2d_filtered = [box for box in boxes_2d if box is not None]
    # Load the image for inference
    image_path = NUSCENES_ROOT / sample_data_cam["filename"]
    image = Image.open(image_path)
    # Create canvas for plotting the results
    category_groups = set([v['category_group'] for v in CATEGORY_MAPPING_TO_UNIAD.values()])
    num_cols = 1 + len(BOX_THRESHOLDS)  # +1 for the ground truth boxes
    num_rows = len(category_groups)  # Number of category groups
    fig, axes = plt.subplots(nrows=num_rows, ncols=num_cols, figsize=(5 * num_cols, 4 * num_rows))
    # Iterate the categories group
    for category_group_index, category_group in enumerate(category_groups):
        # Get the ground truth boxes for the category group
        category_names_in_group = set([v['category_name'] for v in CATEGORY_MAPPING_TO_UNIAD.values() 
                                       if v['category_group'] == category_group])
        ground_truth_boxes_in_group = [
            box for box in boxes_2d_filtered if box.label in category_names_in_group
        ]
        num_gt_boxes_in_group = len(ground_truth_boxes_in_group)
        print(f"Category Group: {category_group}, Categories: {category_names_in_group}, Number of GT boxes: {num_gt_boxes_in_group}")
        # Plot the ground truth boxes for the category group
        category_colors = {category_name: TABLEAU10_NAMES[i % len(TABLEAU10_NAMES)]
                           for i, category_name in enumerate(category_names_in_group)}
        plot_2d_boxes_on_image(image, ground_truth_boxes_in_group,
                               ax=axes[category_group_index, 0],
                               color=category_colors,
                               title=f"{category_group}, GT boxes")
        # Iterate over different box thresholds
        for box_threshold_index, box_threshold in enumerate(BOX_THRESHOLDS):
            # Infer the image with GroundingDINO model
            predicted_boxes, caption = predict_multi_labels(
                model=model,
                image=image,
                labels=category_names_in_group,
                box_threshold=box_threshold,
            )
            print(f"Detected {len(predicted_boxes)} boxes above the threshold of {box_threshold}")
            # convert box coordinates from normalized to pixel coordinates
            for box in predicted_boxes:
                box.xyxy = box.xyxy * np.array([image.width, image.height, image.width, image.height])
            plot_2d_boxes_on_image(image, predicted_boxes,
                                   ax=axes[category_group_index, box_threshold_index + 1],
                                   color=category_colors,
                                   title=f"{category_group}, box_threshold:{box_threshold}")
    plt.show()